# PRO139 – HMI interactivo (estilo base oficial)

Ejecuta la celda siguiente para iniciar el HMI.

In [1]:

import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

# ============================================================
# PRO139 – Devolución de materiales y repuestos a almacén
# HMI oficial (base estilo PRO130 / PRO134 aprobado)
# ============================================================

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

MOTIVOS_BLOQUEO_PRO139 = [
    "Material dañado / contaminado / no apto",
    "Material no corresponde a reserva / OT",
    "Falta identificación de reserva / OT",
    "No es posible registrar devolución en sistema (SAP indisponible / sin acceso)",
    "Problema con etiquetado (sin impresora / sin transacción / error)",
    "Ubicación no disponible / problema físico en almacén",
    "Otro"
]

NODOS = {
    "T1_necesidad": {
        "type": "task",
        "titulo": "Necesidad de devolución",
        "rol": "Solicitante de materiales y repuestos",
        "descripcion": "Se identifica en terreno la necesidad de devolver materiales/repuestos no utilizados al almacén.",
        "acciones": [
            "Identificar material/repuesto no utilizado a devolver."
        ],
        "checklist": [
            "Material/repuesto identificado para devolución"
        ],
        "validacion": "¿Existe material/repuesto no utilizado que requiere devolución al almacén?",
        "next": "T2_verificar_estado"
    },
    "T2_verificar_estado": {
        "type": "task",
        "titulo": "Verificar estado de material/repuesto",
        "rol": "Solicitante",
        "descripcion": "Se verifica condición del material/repuesto previo a su devolución.",
        "acciones": [
            "Verificar condición física (daño, contaminación, uso).",
            "Confirmar que corresponde al material/repuesto retirado."
        ],
        "checklist": [
            "Condición física verificada",
            "Correspondencia validada (material correcto)"
        ],
        "validacion": "¿El material/repuesto está en condición conocida y corresponde a lo retirado?",
        "next": "D1_apto_devolucion"
    },
    "D1_apto_devolucion": {
        "type": "decision",
        "titulo": "¿El material está en estado apto para devolución?",
        "rol": "Solicitante",
        "descripcion": "Decisión sobre si el material puede devolverse al almacén según su condición.",
        "pregunta": "¿El material está en estado apto para devolución?",
        "opciones": [
            {
                "label": "SÍ",
                "next": "T3_identificar_reserva"
            },
            {
                "label": "NO",
                "next": "END_DESECHO"
            }
        ]
    },
    "T3_identificar_reserva": {
        "type": "task",
        "titulo": "Identificar reserva solicitada",
        "rol": "Solicitante",
        "descripcion": "Se identifica la reserva/OT asociada al material para asegurar trazabilidad.",
        "acciones": [
            "Identificar reserva y/o OT asociada al material a devolver."
        ],
        "checklist": [
            "Reserva/OT identificada"
        ],
        "validacion": "¿La reserva/OT asociada al material fue identificada correctamente?",
        "next": "T4_trasladar_almacen"
    },
    "T4_trasladar_almacen": {
        "type": "task",
        "titulo": "Trasladar material/repuesto a almacén",
        "rol": "Solicitante",
        "descripcion": "Se traslada físicamente el material/repuesto al almacén para su recepción.",
        "acciones": [
            "Trasladar material/repuesto al almacén.",
            "Entregar físicamente al especialista de almacén."
        ],
        "checklist": [
            "Material/repuesto trasladado",
            "Entrega física realizada en almacén"
        ],
        "validacion": "¿El material/repuesto fue entregado físicamente al almacén?",
        "next": "T5_revision_almacen"
    },
    "T5_revision_almacen": {
        "type": "task",
        "titulo": "Realizar revisión del material / repuesto",
        "rol": "Especialista de almacén",
        "descripcion": "El especialista de almacén revisa material/repuesto recibido para decidir su almacenaje.",
        "acciones": [
            "Revisar condición física del material/repuesto.",
            "Confirmar correspondencia con reserva/OT indicada."
        ],
        "checklist": [
            "Condición física revisada",
            "Correspondencia con reserva/OT confirmada"
        ],
        "validacion": "¿Se revisó el material/repuesto y corresponde a la reserva/OT indicada?",
        "next": "D2_apto_almacenar"
    },
    "D2_apto_almacenar": {
        "type": "decision",
        "titulo": "¿Material en estado para ser almacenado?",
        "rol": "Especialista de almacén",
        "descripcion": "Decisión de almacén sobre si el material puede volver a stock.",
        "pregunta": "¿Material en estado para ser almacenado?",
        "opciones": [
            {
                "label": "SÍ",
                "next": "T6_registrar_devolucion"
            },
            {
                "label": "NO",
                "next": "END_VENTAS_VARIAS"
            }
        ]
    },
    "T6_registrar_devolucion": {
        "type": "task",
        "titulo": "Registrar devolución de material / repuesto a almacén",
        "rol": "Especialista de almacén",
        "descripcion": "Se registra la devolución en sistema para mantener control de inventario.",
        "acciones": [
            "Registrar devolución de material/repuesto en el sistema (MIGO).",
            "Si aplica: anular reserva asociada (regularización)."
        ],
        "checklist": [
            "Devolución registrada en sistema (MIGO)",
            "Si aplica: reserva anulada"
        ],
        "validacion": "¿La devolución quedó registrada en el sistema y (si aplica) la reserva quedó anulada?",
        "next": "T7_realizar_etiqueta"
    },
    "T7_realizar_etiqueta": {
        "type": "task",
        "titulo": "Realizar etiqueta de material",
        "rol": "Especialista de almacén",
        "descripcion": "Se genera la etiqueta del material para identificación y almacenamiento.",
        "acciones": [
            "Generar etiqueta del material (ZMM_IMP_ETIQUETA / ZMM_MART_CARACT según aplique)."
        ],
        "checklist": [
            "Etiqueta generada"
        ],
        "validacion": "¿La etiqueta del material fue generada correctamente?",
        "next": "D3_con_etiqueta"
    },
    "D3_con_etiqueta": {
        "type": "decision",
        "titulo": "¿Material devuelto con etiqueta?",
        "rol": "Especialista de almacén",
        "descripcion": "Control de que el material está efectivamente etiquetado.",
        "pregunta": "¿Material devuelto con etiqueta?",
        "opciones": [
            {
                "label": "SÍ",
                "next": "T8b_registrar_entrega"
            },
            {
                "label": "NO",
                "next": "T8_etiquetar"
            }
        ]
    },
    "T8_etiquetar": {
        "type": "task",
        "titulo": "Etiquetar material y repuesto",
        "rol": "Especialista de almacén",
        "descripcion": "Se asegura que el material y repuesto queden etiquetados antes de su ubicación.",
        "acciones": [
            "Etiquetar material y repuesto."
        ],
        "checklist": [
            "Material/repuesto etiquetado"
        ],
        "validacion": "¿El material/repuesto quedó correctamente etiquetado?",
        "next": "T8b_registrar_entrega"
    },
    "T9_ubicar": {
        "type": "task",
        "titulo": "Ubicar material / repuesto en el almacén",
        "rol": "Especialista de almacén",
        "descripcion": "Se ubica el material/repuesto en su ubicación definitiva (idealmente dentro de 24 hrs desde el etiquetado).",
        "acciones": [
            "Ubicar material/repuesto en estantería/posición definitiva del almacén.",
            "Registrar ubicación si corresponde."
        ],
        "checklist": [
            "Ubicación definida",
            "Material/repuesto ubicado en almacén",
            "Ubicación registrada si aplica"
        ],
        "validacion": "¿El material/repuesto quedó ubicado en su ubicación definitiva (dentro de 24 hrs desde etiquetado)?",
        "next": "T10_fin_stock"
    },
    "T10_fin_stock": {
        "type": "task",
        "titulo": "Material / repuesto almacenado en ubicación definitiva",
        "rol": "Especialista de almacén",
        "descripcion": "El material/repuesto queda almacenado en su ubicación definitiva. Fin del proceso.",
        "acciones": [
            "Confirmar material/repuesto en ubicación definitiva."
        ],
        "checklist": [
            "Confirmación de almacenamiento realizada"
        ],
        "validacion": "¿El material/repuesto quedó almacenado en ubicación definitiva?",
        "next": "END_OK"
    },
    "END_DESECHO": {
        "type": "end",
        "titulo": "🏁 Proceso finalizado",
        "rol": "HMI",
        "descripcion": "El material no está apto para devolución. Se gestiona como desecho según estándar local.",
        "mensaje": "Ya está en el final/cierre (salida por NO apto para devolución).",
        "estado_final": "FINALIZADO"
    },
    "END_VENTAS_VARIAS": {
        "type": "end",
        "titulo": "🏁 Proceso finalizado",
        "rol": "HMI",
        "descripcion": "El material no está en estado para ser almacenado. Se deriva a Gestión de ventas varias.",
        "mensaje": "Ya está en el final/cierre (salida por NO apto para almacenar).",
        "estado_final": "FINALIZADO"
    },
    "END_OK": {
        "type": "end",
        "titulo": "🏁 Proceso finalizado",
        "rol": "HMI",
        "descripcion": "Se completaron los pasos del PRO139. Puede exportar el JSON auditable si lo requiere.",
        "mensaje": "Ya está en el final/cierre del procedimiento.",
        "estado_final": "FINALIZADO"
    },
    "T8b_registrar_entrega": {
        "type": "task",
        "titulo": "Registrar entrega (Etiqueta + Observaciones)",
        "rol": "Especialista de almacén",
        "descripcion": "Registrar el número de etiqueta y observaciones/comentario sobre la entrega.",
        "acciones": [
            "Registrar el número de etiqueta.",
            "Registrar observaciones/comentario (si aplica)."
        ],
        "inputs": [
            {
                "key": "numero_etiqueta",
                "label": "Ingrese número de etiqueta",
                "required": true
            },
            {
                "key": "comentario_entrega",
                "label": "Observaciones / comentario sobre la entrega (opcional)",
                "required": false,
                "multiline": true
            }
        ],
        "checklist": [
            "Entrega registrada"
        ],
        "validacion": "¿Se registró el número de etiqueta y la entrega quedó registrada?",
        "next": "T9_ubicar"
    }
}


class PRO139HMI:
    def __init__(self):
        self.nodo_id = list(NODOS.keys())[0]
        self.historial = []
        self.logs = []
        self.decisiones = []
        self.inputs = {}  # inputs persistentes (p.ej., número etiqueta, observaciones)
        self._input_widgets = []  # widgets de inputs del nodo actual
        self.bloqueos = []

        self.run_id = str(uuid.uuid4())
        self.estado = EN_CURSO
        self.start_ts = _now_iso()
        self.end_ts = None

        self.output = widgets.Output(layout={"width":"100%"})

        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])

        self.is_blocked = False
        self.block_panel = widgets.VBox([])
        self.btn_rehacer = widgets.Button(description="🔄 Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._decision_widget = None
        self._check_widgets = []

        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "estado": self.estado,
            "nodo": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self):
        self.historial.append(self.nodo_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    def _render_header(self, n):
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;color:black;'><b>ROL:</b> {n.get('rol','')}</span>"
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO139</b> – Devolución de materiales y repuestos a almacén</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        acciones = "".join([f"<li style='margin:4px 0;color:#0f172a;'>{a}</li>" for a in n.get("acciones",[])])
        valid = n.get("validacion","")

        self._check_widgets = [widgets.Checkbox(description=item, value=False) for item in (n.get("checklist",[]) or [])]

        accion_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>⚙️ ACCIÓN A EJECUTAR (texto PRO139)</b></div>
                <ul style="margin-top:10px;padding-left:18px;color:#0f172a;">{acciones}</ul>
            </div>
        """)

        checklist_box = widgets.VBox([])
        if self._check_widgets:
            checklist_box = widgets.VBox([
                widgets.HTML("""
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 CHECKLIST (obligatorio para avanzar)</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Marca cada ítem al completar en terreno.</div>
                </div>
                """),
                widgets.VBox(self._check_widgets)
            ])

        
# Inputs (si el nodo define inputs)
self._input_widgets = []
input_box = widgets.VBox([])
if n.get("inputs"):
    inputs_widgets = []
    for spec in n.get("inputs") or []:
        label = spec.get("label", spec.get("key","Campo")) + ":"
        multiline = bool(spec.get("multiline", False))
        if multiline:
            w = widgets.Textarea(description=label, layout=widgets.Layout(width="100%", height="90px"),
                                 style={"description_width":"initial"})
        else:
            w = widgets.Text(description=label, layout=widgets.Layout(width="100%"),
                             style={"description_width":"initial"})
        # precarga si ya existe
        key = spec.get("key")
        if key in self.inputs:
            w.value = self.inputs.get(key,"")
        self._input_widgets.append((spec, w))
        inputs_widgets.append(w)

    input_box = widgets.VBox([
        widgets.HTML("""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
            <div style="font-size:13px;color:#0f172a;"><b>✍️ INPUTS</b></div>
            <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">Completa los campos requeridos para avanzar.</div>
        </div>
        """),
        widgets.VBox(inputs_widgets)
    ])
valid_box = widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>✅ ¡VALIDACIÓN!</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SÍ</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
            </div>
        """)

        return widgets.VBox([accion_box, checklist_box, input_box, valid_box])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
            </div>
            """),
            radios
        ]), radios

    def _render_block_panel(self):
        if not self.is_blocked:
            self.block_panel.children = []
            return

        title = widgets.HTML("""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #ef4444;background:#fff1f2;">
            <div style="font-size:14px;color:#0f172a;"><b>⛔ BLOQUEADO</b> — Seleccione motivo(s) y registre detalle.</div>
            <div style="margin-top:8px;font-size:12px;color:#0f172a;">No puede avanzar hasta rehacer el paso.</div>
        </div>
        """)

        self.sel_motivos = widgets.SelectMultiple(options=MOTIVOS_BLOQUEO_PRO139, rows=7, layout={"width":"100%"})
        self.txt_detalle = widgets.Textarea(
            placeholder="Detalle del bloqueo (obligatorio si selecciona 'Otro').",
            layout=widgets.Layout(width="100%", height="80px")
        )

        self.block_panel.children = [
            title,
            widgets.HTML("<b>Motivo(s) de bloqueo:</b> (selección múltiple)"),
            self.sel_motivos,
            widgets.HTML("<b>Detalle:</b>"),
            self.txt_detalle,
            self.btn_rehacer
        ]

    def _render_footer(self):
        self.btn_volver.disabled = (len(self.historial) == 0)
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()
            self._clear_msg()

            n = NODOS[self.nodo_id]
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
                self._check_widgets = []
            elif n["type"] == "end":
    self._decision_widget = None
    self._check_widgets = []
    # Resumen visual final
    body_html = f"""
    <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
        <div style="font-size:20px;color:#0f172a;"><b>🏁 FIN</b></div>
        <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','')}</div>
        <div style="margin-top:10px;font-size:12px;color:#0f172a;">Estado final: <b>{n.get('estado_final','')}</b></div>
    </div>
    """
    # Tablas solo en END_OK
    if self.nodo_id == "END_OK":
        # Inputs
        rows_inputs = ""
        if getattr(self, "inputs", None):
            for k, v in self.inputs.items():
                rows_inputs += f"<tr><td style='padding:6px;border:1px solid #e2e8f0;'><b>{k}</b></td><td style='padding:6px;border:1px solid #e2e8f0;'>{(v or '')}</td></tr>"
        if not rows_inputs:
            rows_inputs = "<tr><td colspan='2' style='padding:6px;border:1px solid #e2e8f0;'>(sin inputs)</td></tr>"

        tabla_inputs = f"""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
            <div style="font-size:13px;color:#0f172a;"><b>📌 Inputs registrados</b></div>
            <table style="width:100%;border-collapse:collapse;margin-top:10px;font-size:12px;color:#0f172a;">
                <thead>
                    <tr>
                        <th style="text-align:left;padding:6px;border:1px solid #e2e8f0;background:#f1f5f9;">Campo</th>
                        <th style="text-align:left;padding:6px;border:1px solid #e2e8f0;background:#f1f5f9;">Valor</th>
                    </tr>
                </thead>
                <tbody>{rows_inputs}</tbody>
            </table>
        </div>
        """

        # Decisiones
        rows_dec = ""
        if self.decisiones:
            for d in self.decisiones:
                rows_dec += f"<tr><td style='padding:6px;border:1px solid #e2e8f0;'>{d.get('ts','')}</td><td style='padding:6px;border:1px solid #e2e8f0;'>{d.get('titulo','')}</td><td style='padding:6px;border:1px solid #e2e8f0;'>{d.get('seleccion','')}</td></tr>"
        if not rows_dec:
            rows_dec = "<tr><td colspan='3' style='padding:6px;border:1px solid #e2e8f0;'>(sin decisiones)</td></tr>"

        tabla_dec = f"""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
            <div style="font-size:13px;color:#0f172a;"><b>🧭 Decisiones</b></div>
            <table style="width:100%;border-collapse:collapse;margin-top:10px;font-size:12px;color:#0f172a;">
                <thead>
                    <tr>
                        <th style="text-align:left;padding:6px;border:1px solid #e2e8f0;background:#f1f5f9;">Timestamp</th>
                        <th style="text-align:left;padding:6px;border:1px solid #e2e8f0;background:#f1f5f9;">Nodo</th>
                        <th style="text-align:left;padding:6px;border:1px solid #e2e8f0;background:#f1f5f9;">Selección</th>
                    </tr>
                </thead>
                <tbody>{rows_dec}</tbody>
            </table>
        </div>
        """

        body = widgets.VBox([widgets.HTML(body_html), widgets.HTML(tabla_inputs), widgets.HTML(tabla_dec)])
    else:
        body = widgets.HTML(body_html)

            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")
                self._decision_widget = None
                self._check_widgets = []

            self._render_block_panel()
            footer = self._render_footer()
            self.main_box.children = [header, body, footer]
            display(self.main_box)

    def _check_ready_to_advance(self):
        n = NODOS[self.nodo_id]

        if self.is_blocked:
            return False, "Paso bloqueado. Registre motivo(s) y use 'Rehacer paso'."

        if n["type"] == "end":
            return True, ""

        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                return False, "Debe seleccionar una opción para avanzar."
            return True, ""

        if n["type"] == "task":
    # Checklist: solo obligatorios (los que NO contienen "si aplica" en cualquier parte del texto)
    if self._check_widgets:
        obligatorios = []
        for cb in self._check_widgets:
            txt = (cb.description or "")
            if "si aplica" not in txt.lower():
                obligatorios.append(cb)
        if obligatorios and not all(cb.value for cb in obligatorios):
            return False, "Debe completar el checklist obligatorio antes de avanzar."

    # Inputs requeridos
    if getattr(self, "_input_widgets", None):
        for spec, w in self._input_widgets:
            key = spec.get("key")
            required = bool(spec.get("required", False))
            val = (w.value or "").strip()
            if required and not val:
                return False, f"Debe completar el campo obligatorio: {spec.get('label', key)}"
            if key:
                self.inputs[key] = val

    return True, ""

        return True, ""

    def _advance_to(self, next_id):
        if next_id not in NODOS:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'>
                <b>⚠ Error de flujo:</b> el nodo destino no existe: <code>{next_id}</code>
            </div>
            """)
            self._log("ERROR_FLUJO", {"missing_next": next_id})
            return
        self.nodo_id = next_id
        self._render()

    def _on_si(self, _):
        ok, msg = self._check_ready_to_advance()
        if not ok:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ {msg}</b>
            </div>
            """)
            self._log("VALIDACION_FALLA", {"mensaje": msg})
            return

        n = NODOS[self.nodo_id]

        if n["type"] == "end":
            self.estado = FINALIZADO
            self.end_ts = _now_iso()
            self._log("FINALIZA")
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
                <b>🏁 Ya está en el final/cierre.</b>
            </div>
            """)
            return

        self._push_hist()

        if n["type"] == "task":
            self._log("AVANZA", {"next": n.get("next")})
            self._advance_to(n.get("next"))
        elif n["type"] == "decision":
            chosen_next = self._decision_widget.value
            chosen_label = next((o["label"] for o in n.get("opciones",[]) if o["next"] == chosen_next), None)

            self.decisiones.append({
                "ts": _now_iso(),
                "nodo": self.nodo_id,
                "titulo": n.get("titulo",""),
                "seleccion": chosen_label,
                "next": chosen_next
            })
            self._log("DECISION", {"seleccion": chosen_label, "next": chosen_next})
            self._advance_to(chosen_next)

    def _on_no(self, _):
        if self.is_blocked:
            return
        self.is_blocked = True
        self.estado = BLOQUEADO
        self.block_ts_inicio = _now_iso()
        self._log("BLOQUEADO_INICIO")
        self._render()

    def _on_rehacer(self, _):
        motivos = list(self.sel_motivos.value) if hasattr(self, "sel_motivos") else []
        detalle = (self.txt_detalle.value or "").strip() if hasattr(self, "txt_detalle") else ""

        if not motivos:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe seleccionar al menos un motivo.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "sin_motivo"})
            return

        if "Otro" in motivos and not detalle:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe ingresar detalle si selecciona 'Otro'.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "otro_sin_detalle"})
            return

        bloqueo = {
            "ts_inicio": getattr(self, "block_ts_inicio", None),
            "ts_fin": _now_iso(),
            "nodo": self.nodo_id,
            "titulo": NODOS[self.nodo_id].get("titulo",""),
            "motivos": motivos,
            "detalle": detalle
        }
        self.bloqueos.append(bloqueo)
        self._log("BLOQUEADO_FIN", bloqueo)

        self.is_blocked = False
        self.estado = EN_CURSO
        self._log("REHACER_PASO")
        self._render()

    def _on_volver(self, _):
        if self.is_blocked:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ No puede volver mientras el paso está bloqueado. Use 'Rehacer paso'.</b>
            </div>
            """)
            return

        prev_id = self._pop_hist()
        if prev_id is not None:
            self._log("VOLVER", {"to": prev_id})
            self._advance_to(prev_id)

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO139 – Devolución de materiales y repuestos a almacén",
            "run_id": self.run_id,
            "estado": self.estado,
            "start_ts": self.start_ts,
            "end_ts": self.end_ts,
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "decisiones": list(self.decisiones),
            "bloqueos": list(self.bloqueos),
            "logs": list(self.logs),
            "inputs": dict(getattr(self, "inputs", {})),
            "export_ts": _now_iso(),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(self.output)

hmi = PRO139HMI()
hmi.iniciar()


Output(layout=Layout(width='100%'))